# Comparación de Yahoo Finance y Financial Modeling Prep

Este notebook comprueba que **ambas APIs devuelven objetos con la misma estructura**, muestra precios y estados financieros y compara los ejercicios disponibles. **No presupone que ambas fuentes tengan cifras idénticas.**

**Ubicación recomendada:** `Financial-Toolkit/tests/comparacion_yahoo_fmp_tests.ipynb`. La primera celda de código localiza `src/models`, configura las importaciones y carga `FMP_API_KEY` desde el `.env` de la raíz. Funciona con el kernel iniciado desde la raíz del proyecto o desde `tests/`.


## 1. Importaciones y clave de FMP

In [ ]:
# Si faltan dependencias, ejecuta esta línea una sola vez:
# %pip install pandas numpy yfinance requests python-dotenv

import os
import sys
from pathlib import Path
from dataclasses import fields, is_dataclass
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

# Compatible con un kernel iniciado desde tests/ o desde la raíz del repositorio.
# También busca la raíz en los directorios superiores del directorio de trabajo.
working_dir = Path.cwd().resolve()
project_root = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / "src" / "models").is_dir()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        "No se encuentra src/models. Abre Financial-Toolkit en VS Code "
        "y ejecuta el notebook desde la raíz o desde tests/."
    )

models_path = project_root / "src" / "models"
if str(models_path) not in sys.path:
    sys.path.insert(0, str(models_path))

load_dotenv(project_root / ".env")
api_key = os.getenv("FMP_API_KEY")
if not api_key:
    raise ValueError(f"Falta FMP_API_KEY en {project_root / '.env'}")
print(f"Raíz del proyecto: {project_root}")
print(f"Modelos disponibles en: {models_path}")
print("Clave FMP cargada correctamente (no se muestra).")


## 2. Crear las dos APIs y descargar la misma empresa

In [ ]:
# Importar las implementaciones reales desde src/models (ruta configurada arriba).
from YahooFinanceAPI import YahooFinanceAPI
from fmp_api import FMPAPI

yahoo_api = YahooFinanceAPI()
fmp_api = FMPAPI(api_key=api_key)

ticker = "AAPL"
stock_yahoo = yahoo_api.get_stock(ticker, period="5y", interval="1d")
stock_fmp = fmp_api.get_stock(ticker, period="5y", interval="1d")

print("Yahoo:", stock_yahoo.company.name)
print("FMP:", stock_fmp.company.name)


## 3. Comprobar tipos y estructura

In [ ]:
print("Tipos de Stock:", type(stock_yahoo).__name__, type(stock_fmp).__name__)
assert type(stock_yahoo) is type(stock_fmp), "Las APIs no devuelven la misma clase Stock"

for label, a, b in [
    ("Stock", stock_yahoo, stock_fmp),
    ("PriceSeries", stock_yahoo.price, stock_fmp.price),
    ("Company", stock_yahoo.company, stock_fmp.company),
    ("Fundamentals", stock_yahoo.company.fundamentals, stock_fmp.company.fundamentals),
]:
    assert type(a) is type(b), f"Distinto tipo en {label}"
    print(f"{label}: OK ({type(a).__name__})")

for name in ["income_statements", "balance_sheets", "cash_flows"]:
    a = getattr(stock_yahoo.company.fundamentals, name)
    b = getattr(stock_fmp.company.fundamentals, name)
    assert type(a) is type(b), f"Distinto tipo en {name}"
    print(f"{name}: OK ({type(a).__name__})")

if is_dataclass(stock_yahoo):
    print("Atributos de Stock:", [f.name for f in fields(stock_yahoo)])

## 4. Comprobar precios y verlos en tablas

La versión actual de `PriceSeries` utiliza atributos separados (`date`, `open`, `high`, `low`, `close`, `volume`).

In [ ]:
for nombre, stock in [("Yahoo", stock_yahoo), ("FMP", stock_fmp)]:
    p = stock.price
    print(f"{nombre}")
    for atributo in ["date", "open", "high", "low", "close", "volume"]:
        assert hasattr(p, atributo), f"Falta {atributo} en {nombre}"
    print("Observaciones:", len(p.close))
    print("Último cierre:", p.close.iloc[-1])
    if hasattr(p, "to_dataframe"):
        display(p.to_dataframe().tail())
    else:
        display(pd.DataFrame({
            "Open": p.open.to_numpy(), "High": p.high.to_numpy(),
            "Low": p.low.to_numpy(), "Close": p.close.to_numpy(),
            "Volume": p.volume.to_numpy()
        }, index=p.date).tail())

## 5. Comparar los cierres en las mismas fechas

**Atención:** Yahoo usa `auto_adjust=True`; FMP puede aplicar otro criterio de ajuste. Diferencias en el cierre no implican necesariamente un error.

In [ ]:
yahoo_close = stock_yahoo.price.close.copy()
fmp_close = stock_fmp.price.close.copy()

yahoo_close.index = yahoo_close.index.date
fmp_close.index = fmp_close.index.date

cierres = pd.concat([
    yahoo_close.rename("Yahoo"),
    fmp_close.rename("FMP")
], axis=1).dropna()

cierres["Diferencia"] = cierres["Yahoo"] - cierres["FMP"]
cierres["Diferencia_%"] = (cierres["Diferencia"] / cierres["FMP"]) * 100

print("Fechas comunes:", len(cierres))

display(cierres.tail(10))

if not cierres.empty:
    cierres[["Yahoo", "FMP"]].plot(
        figsize=(12, 5),
        title=f"{ticker}: cierres por proveedor",
        grid=True
    )

## 6. Comparar los estados financieros disponibles

In [ ]:
resultados = []
for nombre in ["income_statements", "balance_sheets", "cash_flows"]:
    a = getattr(stock_yahoo.company.fundamentals, nombre)
    b = getattr(stock_fmp.company.fundamentals, nombre)
    assert hasattr(a, "statements") and hasattr(b, "statements"), "Se esperaba el atributo statements"
    resultados.append({
        "Estado": nombre,
        "Ejercicios Yahoo": len(a.statements),
        "Ejercicios FMP": len(b.statements),
        "Fechas Yahoo": sorted(str(s.date) for s in a.statements),
        "Fechas FMP": sorted(str(s.date) for s in b.statements),
    })
display(pd.DataFrame(resultados))

## 7. Comprobar que cada estado financiero tiene los mismos atributos

In [ ]:
for nombre in ["income_statements", "balance_sheets", "cash_flows"]:
    a = getattr(stock_yahoo.company.fundamentals, nombre).statements
    b = getattr(stock_fmp.company.fundamentals, nombre).statements
    if a and b:
        assert type(a[0]) is type(b[0]), f"Clases diferentes en {nombre}"
        atributos_a = [f.name for f in fields(a[0])]
        atributos_b = [f.name for f in fields(b[0])]
        assert atributos_a == atributos_b
        print(f"{nombre}: OK → {len(atributos_a)} atributos; clase {type(a[0]).__name__}")
    else:
        print(f"{nombre}: sin datos suficientes en alguna API para comparar")

## 8. `get_by_year()` y `to_dataframe()` (si ya los añadiste)

In [ ]:
for nombre, stock in [("Yahoo", stock_yahoo), ("FMP", stock_fmp)]:

    print(f"===== {nombre} =====")

    history = stock.company.fundamentals.income_statements

    df = history.to_dataframe()

    if df.empty:
        print("No hay estados financieros disponibles")

    else:
        display(df.iloc[:8, :5])

        year = df.columns[0].year

        print(f"get_by_year({year}):")
        print(history.get_by_year(year))

## 9. Ejecutar los análisis de `Fundamentals`

Se ejecuta cada análisis por separado para que puedas localizar errores en los cálculos si falta algún dato contable.

In [ ]:
for nombre, stock in [("Yahoo", stock_yahoo), ("FMP", stock_fmp)]:
    print(f"===== ANÁLISIS {nombre} =====")
    fundamentals = stock.company.fundamentals
    for etiqueta, metodo in [
        ("Income Statement", fundamentals.analyze_income_statement),
        ("Balance Sheet", fundamentals.analyze_balance_sheet),
        ("Cash Flow", fundamentals.analyze_cash_flow),
    ]:
        print(etiqueta)
        # Si hay divisiones por cero o datos NaN, revisa la definición de los campos.
        df = metodo()
        assert isinstance(df, pd.DataFrame)
        display(df.head())

## 10. Comparar una partida contable por ejercicio

La comparación se hace por **año fiscal** y solo para años presentes en ambas fuentes. Antes de interpretar diferencias, verifica moneda, definición de la partida y fecha de cierre.

In [ ]:
filas = []
for nombre, atributo in [
    ("income_statements", "total_revenue"),
    ("income_statements", "net_income"),
    ("balance_sheets", "total_assets"),
    ("cash_flows", "operating_cash_flow"),
]:
    h_y = getattr(stock_yahoo.company.fundamentals, nombre).statements
    h_f = getattr(stock_fmp.company.fundamentals, nombre).statements
    por_anio_y = {s.date.year: s for s in h_y}
    por_anio_f = {s.date.year: s for s in h_f}
    for year in sorted(set(por_anio_y) & set(por_anio_f)):
        y = getattr(por_anio_y[year], atributo)
        f = getattr(por_anio_f[year], atributo)
        filas.append({"Estado": nombre, "Partida": atributo, "Año": year,
                      "Yahoo": y, "FMP": f, "Diferencia": y - f})
comparacion = pd.DataFrame(filas)
if comparacion.empty:
    print("No hay años fiscales comunes para comparar")
else:
    display(comparacion)

## Interpretación

- **Mismo tipo de objeto y atributos:** demuestra que tus adaptadores ofrecen una interfaz común.
- **Diferencias en precios:** comprueba primero el ajuste por dividendos/splits.
- **Diferencias en estados financieros:** comprueba moneda, fecha fiscal, signos y equivalencia contable de los campos.
- **Faltan datos en FMP:** puede depender de tu plan de API o de la cobertura del ticker.

**Importante:** que dos objetos sean comparables estructuralmente **no garantiza** que sus cifras sean económicamente equivalentes. Esa segunda comprobación requiere revisar el mapeo de campos de ambas APIs.